In [3]:
import pandas as pd
from datetime import datetime

In [20]:
laboratory_result=pd.read_json('laboratory_result.json')

In [21]:
laboratory_result.head()

,result_reported,pcr_lab_sample_number,checked_by,result_report,date_approved,uuid,date_sample_received_at_pcr_lab,date_result_received,archived,patient_uuid,...,result_reported_by,date_created,created_by,approved_by,assayed_by,date_modified,patient_id,date_checked,modified_by,pcr_lab_name
0,0,,,,,8f52ff62-fe34-4460-ab8a-f598a8a6ad49,,2022-05-23 00:00:00.0,0,aa4705ad-f504-4bfe-824b-28219eaed5e0,...,,2022-05-23 14:28:30.243,ETL,,,2022-05-23 14:28:30.243,1691,,ETL,
1,13434,,,,,3f782265-f321-43a1-be57-ebe77653abf4,,2022-05-23 00:00:00.0,0,07a38ff0-e53a-4c2e-9151-4056a39e0700,...,,2022-07-07 17:16:44.152,ETL,,,2022-07-07 17:16:44.152,2826,,ETL,
2,0,,,,,dead6437-c4ca-44df-837f-1e9ead644667,,2022-05-23 00:00:00.0,0,77afc9c2-82eb-45b0-a136-6c39471a584f,...,,2022-05-23 14:19:05.157,ETL,,,2022-05-23 14:19:05.157,2837,,ETL,
3,0,,,,,1727effb-7473-46e1-ac4f-626b2c71ef16,,2022-05-23 00:00:00.0,0,aad8e176-2a65-4163-b6c7-49b01ed9c610,...,,2022-05-23 14:18:35.861,ETL,,,2022-05-23 14:18:35.861,2893,,ETL,
4,0,,,,,18b80df8-bc92-4311-a94a-46877f78f174,,2022-05-23 00:00:00.0,0,e648d268-28a9-440e-a753-0d2af58a7bb4,...,,2022-05-23 14:29:33.934,ETL,,,2022-05-23 14:29:33.934,1456,,ETL,


In [66]:
indexes = [1032,1045]
laboratory_result.drop(indexes)
laboratory_result.loc[indexes, :]

,result_reported,pcr_lab_sample_number,checked_by,result_report,date_approved,uuid,date_sample_received_at_pcr_lab,date_result_received,archived,patient_uuid,...,result_reported_by,date_created,created_by,approved_by,assayed_by,date_modified,patient_id,date_checked,modified_by,pcr_lab_name
1032,484,,,,,e7ae4e1b-c502-4956-96cd-2b0299203c08,,20191-06-19 00:00:00.0,0,44d3d424-e21a-4aed-a39b-25ced23cdf37,...,,2019-09-09 20:47:16.801,ETL,,,2019-09-09 20:47:16.801,1473,,ETL,
1045,,,,,,e7a6429d-305e-4b8b-8bfa-cd929847b0a7,,,0,942e4327-71d7-465f-b38a-fdb73913b776,...,,2024-01-15 12:58:31.17,ecewsACE5,,,2024-01-15 12:58:31.17,1575,,ecewsACE5,


In [3]:
# df = expanded_radet_w21.query('datim_id=="AFtz58N0L6c"')

In [61]:
laboratory_result=pd.read_json('laboratory_result.json')
def _date_validation(df):
        date_columns = [col for col in df.columns if col.startswith('date_') or col.endswith('_date')]
        
        if not date_columns:
            return {}  # No date columns to validate
        
        problematic_dates = {}
        indexes_for_bad_dates = []
        for col in date_columns:
            try:
                pd.to_datetime(df[col], errors='raise')
            except (TypeError, ValueError) as e:
                problematic_dates[col] = []
                for idx, value in df[col].items():
                    try:
                        pd.to_datetime(value, errors='raise')
                    except (TypeError, ValueError):
                        indexes_for_bad_dates.append(idx)
                        problematic_dates[col].append(f'record {idx+1}, value => {value}')
        
        return problematic_dates,indexes_for_bad_dates

validation_result=_date_validation(laboratory_result)
if validation_result[0] != {}:
    laboratory = laboratory_result.drop(validation_result[1])

# laboratory_result.query('date_result_received=="20191-06-19 00:00:00.0"')
laboratory.query('date_result_received=="20191-06-19 00:00:00.0"')

# if validation_result != {}:
#     print(validation_result)
# else:
#     print('passed')

,result_reported,pcr_lab_sample_number,checked_by,result_report,date_approved,uuid,date_sample_received_at_pcr_lab,date_result_received,archived,patient_uuid,...,result_reported_by,date_created,created_by,approved_by,assayed_by,date_modified,patient_id,date_checked,modified_by,pcr_lab_name


In [54]:
def _date_validation(df):
        date_columns = [col for col in df.columns if col.startswith('date_') or col.endswith('_date')]
        
        if not date_columns:
            return {}  # No date columns to validate
        
        problematic_dates = {}
        
        for col in date_columns:
            try:
                pd.to_datetime(df[col], errors='raise')
            except (TypeError, ValueError) as e:
                problematic_dates[col] = []
                for idx, value in df[col].items():
                    try:
                        pd.to_datetime(value, errors='raise')
                    except (TypeError, ValueError):
                        problematic_dates[col].append(f'record {idx+1}, value => {value}')
        
        return problematic_dates

validation_result=_date_validation(laboratory_result)
validation_result
# if validation_result != {}:
#     print(validation_result)
# else:
#     print('passed')

{'date_result_received': ['record 1033, value => 20191-06-19 00:00:00.0']}

In [9]:
def _date_validation(df):   
    date_columns = [col for col in df.columns if col.startswith('date') or col.endswith('date')]
    problematic_dates = {}
    if date_columns:
        df_dates = df[date_columns].fillna('2024-01-01')
        validity_results = {}
        for col in date_columns:
            try:
                pd.to_datetime(df_dates[col])
                validity_results[col] = True  # Date is valid
            except ValueError:
                validity_results[col] = False  # Date is invalid or column contains non-date data
        failed_columns = [key for key, value in validity_results.items() if value is False]
        if failed_columns != []:
            df_with_bad_date_columns=df[failed_columns]
            for column in failed_columns:
                for idx, value in enumerate(df_with_bad_date_columns[column]):
                    try:
                        pd.to_datetime(value, errors='raise')
                    except (TypeError, ValueError):
                        if column not in problematic_dates:
                            problematic_dates[column] = []
                        problematic_dates[column].append((f'record {idx+1}, bad_date_value=> {value}'))
        if problematic_dates != {}:
            return problematic_dates
        else:
            return {}
    else:
        return {}
    
validation_result=_date_validation(expanded_radet_w25)
validation_result

{'dateofstartofcurrentartregimen': ['record 7262, bad_date_value=> ###############################################################################################################################################################################################################################################################',
  'record 44912, bad_date_value=> ###############################################################################################################################################################################################################################################################'],
 'dateofcurrentartstatus': ['record 19858, bad_date_value=> ###############################################################################################################################################################################################################################################################'],
 'previousstatusdate': ['record 19858, bad_date_value=> #######